# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [1]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: c:\Users\User\Documents\GitHub\echochamber-project-team-1
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [3]:
student_id = "student_04"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "student_04_youtube_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [4]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []

with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

df.head()

,id,source_platform,source_channel,text_raw,video_id,video_title,video_date,comment_date,likes,collected_at,text,lang
0,yt_8r0mvpeBZM0_UgytN-JwLVb_Sdj_9kJ4AaABAg,youtube,AdevaruriSecrete,"Deci Putin planuies😮ceva rau, Dar Trump doar ...",8r0mvpeBZM0,Scutul Lui Putin 🇷🇺 #stiri,2026-04-16,2026-04-16,3,2026-05-12,"Deci Putin planuies😮ceva rau, Dar Trump doar i...",ro
1,yt_8r0mvpeBZM0_UgyTTM29RdEDsG3Yi194AaABAg,youtube,AdevaruriSecrete,..ARE PUTERE PT K L A INVESTIT BUNUL DUMNEZEU....,8r0mvpeBZM0,Scutul Lui Putin 🇷🇺 #stiri,2026-04-16,2026-04-21,0,2026-05-12,..ARE PUTERE PT K L A INVESTIT BUNUL DUMNEZEU....,ro
2,yt_8r0mvpeBZM0_UgzunLTMGAsAGyQaULx4AaABAg,youtube,AdevaruriSecrete,"Aia de la putin e servieta cu toarte, nu cu ma...",8r0mvpeBZM0,Scutul Lui Putin 🇷🇺 #stiri,2026-04-16,2026-04-16,0,2026-05-12,"Aia de la putin e servieta cu toarte, nu cu ma...",ro
3,yt_8r0mvpeBZM0_Ugzls62ywMrrDXaHxnF4AaABAg,youtube,AdevaruriSecrete,Ce scut măi nene?! Aia e valiza in care sunt c...,8r0mvpeBZM0,Scutul Lui Putin 🇷🇺 #stiri,2026-04-16,2026-04-16,2,2026-05-12,Ce scut măi nene?! Aia e valiza in care sunt c...,ro
4,yt_qOam31LRc2k_UgwL8zRSoktCn_Y0FKF4AaABAg,youtube,AdevaruriSecrete,"m-am saturat de stirile voastre alarmiste, sa ...",qOam31LRc2k,Europa Va Rămâne Fără Petrol #stiri,2026-04-16,2026-04-16,6,2026-05-12,"m-am saturat de stirile voastre alarmiste, sa ...",ro


In [5]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 18
Columns: ['id', 'source_platform', 'source_channel', 'text_raw', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'collected_at', 'text', 'lang']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [6]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(15) # completează pentru a vedea cele mai frecvente 15 canale sursă din dataset

source_channel
AdevaruriSecrete    18
Name: count, dtype: int64

In [7]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
0,AdevaruriSecrete,Scutul Lui Putin 🇷🇺 #stiri,"Deci Putin planuies😮ceva rau, Dar Trump doar i..."
1,AdevaruriSecrete,Scutul Lui Putin 🇷🇺 #stiri,..ARE PUTERE PT K L A INVESTIT BUNUL DUMNEZEU....
8,AdevaruriSecrete,Criza Globala A Petrolului NE VA LOVI In Curand,Văd că ați uitat cu toți ce declarați făcea WE...
5,AdevaruriSecrete,Europa Va Rămâne Fără Petrol #stiri,Unde este petrolul pe care tara noastra il ave...
3,AdevaruriSecrete,Scutul Lui Putin 🇷🇺 #stiri,Ce scut măi nene?! Aia e valiza in care sunt c...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [8]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
0,AdevaruriSecrete,"Deci Putin planuies😮ceva rau, Dar Trump doar i..."
1,AdevaruriSecrete,..ARE PUTERE PT K L A INVESTIT BUNUL DUMNEZEU....
8,AdevaruriSecrete,Văd că ați uitat cu toți ce declarați făcea WE...
5,AdevaruriSecrete,Unde este petrolul pe care tara noastra il ave...
3,AdevaruriSecrete,Ce scut măi nene?! Aia e valiza in care sunt c...
13,AdevaruriSecrete,O Americă în cădere și cu multe datorii este c...
16,AdevaruriSecrete,Asta este dorința javrelor Sioniste Papusarii ...
15,AdevaruriSecrete,Ce a început? Resetsrea? Cine este la butoane?...
11,AdevaruriSecrete,"Niciodata, nenea trump, vegheaza asupra noastr..."
2,AdevaruriSecrete,"Aia de la putin e servieta cu toarte, nu cu ma..."


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [9]:
SYSTEM_PROMPT = """
Ești un analist specializat în detectarea discursului conspiraționist, suveranist și anti-sistem din comentarii politice online românești.

Trebuie să identifici tiparele de discurs specifice:
- neîncredere în instituții
- acuzații de manipulare sau control din umbră
- conspirații geopolitice
- propagandă pro/anti Rusia, SUA, UE, NATO
- idei despre „sistem”, „globaliști”, „stat paralel”
- insinuări fără dovezi
- discurs alarmist sau anti-elitist

Analizează comentariul exclusiv pe baza textului oferit.
Nu inventa informații care nu există în comentariu.
Răspunde STRICT în format JSON valid.
Nu adăuga explicații suplimentare.
"""

USER_PROMPT_TEMPLATE = """
Citește următorul comentariu politic și identifică:

1. target:
Actorul principal criticat sau menționat.
Exemple:
- guvern
- UE
- NATO
- Rusia
- SUA
- Ucraina
- presa
- politicieni
- CCR
- sistemul
- instituțiile statului

2. stance:
Poziția comentariului față de target.
Valori posibile:
- pro
- anti
- neutru
- ambivalent

3. sentiment:
Sentimentul dominant transmis de comentariu.
Valori posibile:
- negativ
- pozitiv
- frică
- furie
- neîncredere
- sarcasm
- anxietate
- speranță

4. tone:
Tonul discursului.
Exemple:
- conspiraționist
- alarmist
- sarcastic
- agresiv
- ironic
- naționalist
- propagandistic
- pesimist
- suveranist

5. topic:
Tema principală discutată.
Exemple:
- război
- geopolitică
- alegeri
- corupție
- propagandă
- manipulare media
- NATO
- Rusia
- SUA
- Ucraina
- pandemie
- suveranitate

6. interpretation_problem:
Tipul principal de interpretare conspiraționistă sau problemă de raționament observată.
Exemple:
- conspirație fără dovezi
- generalizare excesivă
- frică geopolitică
- manipulare emoțională
- neîncredere instituțională
- propagandă
- polarizare
- demonizarea adversarului
- fake causality
- narativ anti-sistem

7. conspiracy_level:
Nivelul de intensitate conspiraționistă.
Valori posibile:
- scăzut
- mediu
- ridicat

Important:
- Returnează DOAR JSON valid.
- Nu folosi markdown.
- Nu adăuga explicații.
- Folosește exact cheile cerute.
- Dacă o categorie nu este clară, alege cea mai apropiată interpretare.

Returnează JSON valid cu exact aceste chei:
target,
stance,
sentiment,
tone,
topic,
interpretation_problem,
conspiracy_level

Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [14]:
from openai import OpenAI
client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [15]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [17]:
n_comments = 5  # schimbă aici: 3, 5 sau 10
sample_for_prompt = sample_df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,AdevaruriSecrete,Scutul Lui Putin 🇷🇺 #stiri,"Deci Putin planuies😮ceva rau, Dar Trump doar i...","```json\n{\n ""target"": [\n ""Putin"",\n ""..."
1,AdevaruriSecrete,Scutul Lui Putin 🇷🇺 #stiri,..ARE PUTERE PT K L A INVESTIT BUNUL DUMNEZEU....,"{\n ""target"": ""masoneriei"",\n ""stance"": ""ant..."
2,AdevaruriSecrete,Criza Globala A Petrolului NE VA LOVI In Curand,Văd că ați uitat cu toți ce declarați făcea WE...,"{\n ""target"": ""WEF"",\n ""stance"": ""anti"",\n ..."
3,AdevaruriSecrete,Europa Va Rămâne Fără Petrol #stiri,Unde este petrolul pe care tara noastra il ave...,"{\n ""target"": ""instituțiile statului"",\n ""st..."
4,AdevaruriSecrete,Scutul Lui Putin 🇷🇺 #stiri,Ce scut măi nene?! Aia e valiza in care sunt c...,"{\n ""target"": ""necunoscut"",\n ""stance"": ""neu..."


# 9. Verificam rezultatele

In [18]:
results_df.model_output[0]

'```json\n{\n  "target": [\n    "Putin",\n    "Trump"\n  ],\n  "stance": [\n    "anti",\n    "neutru"\n  ],\n  "sentiment": "neîncredere",\n  "tone": [\n    "conspiraționist",\n    "propagandistic"\n  ],\n  "topic": [\n    "geopolitică",\n    "propagandă"\n  ],\n  "interpretation_problem": [\n    "conspirație fără dovezi",\n    "demonizarea adversarului"\n  ],\n  "conspiracy_level": "mediu"\n}\n```'

In [19]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [20]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,AdevaruriSecrete,Scutul Lui Putin 🇷🇺 #stiri,"Deci Putin planuies😮ceva rau, Dar Trump doar i...","[Putin, Trump]","[anti, neutru]",neîncredere,"[conspiraționist, propagandistic]","[geopolitică, propagandă]","[conspirație fără dovezi, demonizarea adversar...",
1,AdevaruriSecrete,Scutul Lui Putin 🇷🇺 #stiri,..ARE PUTERE PT K L A INVESTIT BUNUL DUMNEZEU....,masoneriei,anti,neîncredere,conspiraționist,conspirație,conspirație fără dovezi,
2,AdevaruriSecrete,Criza Globala A Petrolului NE VA LOVI In Curand,Văd că ați uitat cu toți ce declarați făcea WE...,WEF,anti,frică,conspiraționist,noua ordine mondială,conspirație fără dovezi,
3,AdevaruriSecrete,Europa Va Rămâne Fără Petrol #stiri,Unde este petrolul pe care tara noastra il ave...,instituțiile statului,anti,neîncredere,conspiraționist,corupție,conspirație fără dovezi,
4,AdevaruriSecrete,Scutul Lui Putin 🇷🇺 #stiri,Ce scut măi nene?! Aia e valiza in care sunt c...,necunoscut,neutru,sarcasm,sarcastic,manipulare media,conspirație fără dovezi,


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 
Promtul separă destul de bine cele două categorii.

In [23]:
PROJECT_ROOT = Path(r"C:\Users\User\Documents\GitHub\echochamber-project-team-1")

parsed_df.to_csv(
    PROJECT_ROOT / "notebooks" / "student_4" / "parsed_outputs.csv",
    index=False,
    encoding="utf-8-sig"
)